# SANPO Session Kinetic Replay

This notebook takes one processed JSONL session, recomputes per-object kinetic scores with a **custom formula you control**, and renders an MP4 by pulling SANPO RGB frames using `session_id` + `frame_id`.

Important notes:
- Object-level fields in these JSONL files come from the fact-sheet text (`class`, `distance`, `velocity`, direction).
- The JSONL does **not** include per-object bounding boxes, so this replay highlights each object's kinetic score in a ranked on-frame panel (not box overlays).
- You can edit the formula parameters in the config cell.

In [ ]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
from IPython.display import Video, display

In [ ]:
# ---------- User Config ----------
PROJECT_ROOT = Path('.').resolve()

# One processed JSONL session file
JSONL_PATH = PROJECT_ROOT / 'data/processed/vIp8vPcM-fyU2ktEMitwbA2Vsc0A1vtr.jsonl'

# SANPO session folder name under data/sanpo/raw
# Change this to the matching SANPO session id you want to replay.
SESSION_ID = '01cbb9d502fbee1fb172e1740ea9ece2fe35dc6b79c6a9411653132cd9afddcf'

RGB_FRAMES_DIR = PROJECT_ROOT / f'data/sanpo/raw/{SESSION_ID}/camera_head/left/video_frames'
DESCRIPTION_JSON = PROJECT_ROOT / f'data/sanpo/raw/{SESSION_ID}/description.json'
OUTPUT_MP4 = PROJECT_ROOT / f'data/processed/{JSONL_PATH.stem}_{SESSION_ID[:8]}_kinetic_replay.mp4'

# Render options
SCALE = 0.60
DRAW_TOP_N = 8
OUTPUT_FPS = None  # None => infer from session fps / frame stride

# ---------- Custom Kinetic Formula Parameters ----------
# K = severity(class) * direction_weight(position) * |velocity|^VELOCITY_EXP / max(distance, EPS_DISTANCE)^DISTANCE_EXP
EPS_DISTANCE = 0.5
VELOCITY_EXP = 2.0
DISTANCE_EXP = 1.0
UNKNOWN_DISTANCE_SCORE = 0.0
DEFAULT_SEVERITY = 1.0

AHEAD_WEIGHT = 1.00
LATERAL_WEIGHT = 0.85
FAR_LATERAL_WEIGHT = 0.70
REAR_WEIGHT = 0.60
UNKNOWN_DIRECTION_WEIGHT = 0.75

SEVERITY_BY_CLASS = {
    'person': 1.2,
    'bicycle': 1.4,
    'motorcycle': 1.8,
    'car': 2.0,
    'bus': 2.5,
    'truck': 2.5,
    'unlabeled_obstacle': 1.3,
    'fire hydrant': 0.8,
    'shadow': 0.2,
    'boat': 0.5,
}

print(f'JSONL:       {JSONL_PATH}')
print(f'SESSION_ID:  {SESSION_ID}')
print(f'RGB DIR:     {RGB_FRAMES_DIR}')
print(f'OUTPUT MP4:  {OUTPUT_MP4}')

In [ ]:
@dataclass
class ObjectFact:
    object_id: str
    obj_class: str
    distance_m: Optional[float]
    velocity_ms: float
    position: str


OBJECT_PREFIX = re.compile(r'^(Object_\d+):\s*(.+)$')


def parse_distance(token: str) -> Optional[float]:
    token = token.strip().lower()
    m = re.search(r'([0-9]+(?:\.[0-9]+)?)m', token)
    return float(m.group(1)) if m else None


def parse_object_segment(segment: str) -> Optional[ObjectFact]:
    m = OBJECT_PREFIX.match(segment.strip())
    if not m:
        return None

    object_id = m.group(1)
    payload = m.group(2)
    parts = [p.strip() for p in payload.split(',')]
    if len(parts) < 2:
        return None

    obj_class = parts[0]
    distance_m = parse_distance(parts[1])

    vel_match = re.search(r'v=([-+]?\d*\.?\d+)m/s', payload)
    velocity_ms = float(vel_match.group(1)) if vel_match else 0.0

    pos_match = re.search(r'\),\s*([^\[]+)\s*\[', payload)
    position = pos_match.group(1).strip() if pos_match else 'unknown'

    return ObjectFact(
        object_id=object_id,
        obj_class=obj_class,
        distance_m=distance_m,
        velocity_ms=velocity_ms,
        position=position,
    )


def parse_jsonl_record(raw_line: str):
    row = json.loads(raw_line)
    assistant = json.loads(row.get('assistant', '{}'))
    if 'frame_id' not in assistant:
        return None

    frame_id = int(assistant['frame_id'])
    user_text = row.get('user', '')
    fact_text = user_text.split('] ', 1)[1] if '] ' in user_text else user_text

    object_segments = [
        seg.strip()
        for seg in fact_text.split('|')
        if seg.strip().startswith('Object_')
    ]

    objects = []
    for seg in object_segments:
        obj = parse_object_segment(seg)
        if obj is not None:
            objects.append(obj)

    return frame_id, objects


def direction_weight(position: str) -> float:
    p = position.lower()
    if 'ahead' in p:
        return AHEAD_WEIGHT
    if 'far-left' in p or 'far-right' in p:
        return FAR_LATERAL_WEIGHT
    if 'left' in p or 'right' in p:
        return LATERAL_WEIGHT
    if 'behind' in p:
        return REAR_WEIGHT
    return UNKNOWN_DIRECTION_WEIGHT


def kinetic_score_custom(obj: ObjectFact) -> float:
    if obj.distance_m is None:
        return float(UNKNOWN_DISTANCE_SCORE)

    severity = float(SEVERITY_BY_CLASS.get(obj.obj_class, DEFAULT_SEVERITY))
    v = abs(float(obj.velocity_ms))
    d = max(float(obj.distance_m), float(EPS_DISTANCE))

    return float(severity * direction_weight(obj.position) * (v ** VELOCITY_EXP) / (d ** DISTANCE_EXP))


def resolve_frame_path(frame_id: int) -> Optional[Path]:
    for ext in ('.png', '.jpg', '.jpeg'):
        p = RGB_FRAMES_DIR / f'{frame_id:06d}{ext}'
        if p.exists():
            return p
    return None


def infer_session_fps() -> Optional[float]:
    if not DESCRIPTION_JSON.exists():
        return None
    with open(DESCRIPTION_JSON, 'r', encoding='utf-8') as f:
        d = json.load(f)
    cams = d.get('session_camera_details', [])
    if not cams:
        return None
    fps = cams[0].get('fps')
    return float(fps) if fps else None


def infer_stride(frame_ids: list[int]) -> int:
    if len(frame_ids) < 2:
        return 1
    diffs = [b - a for a, b in zip(frame_ids[:-1], frame_ids[1:]) if b > a]
    if not diffs:
        return 1
    return max(1, int(round(float(np.median(np.array(diffs))))))

In [ ]:
if not JSONL_PATH.exists():
    raise FileNotFoundError(f'JSONL file not found: {JSONL_PATH}')

frame_objects: dict[int, list[ObjectFact]] = {}
with open(JSONL_PATH, 'r', encoding='utf-8') as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            continue
        parsed = parse_jsonl_record(line)
        if parsed is None:
            continue
        frame_id, objects = parsed
        frame_objects[frame_id] = objects

frame_ids = sorted(frame_objects.keys())
if not frame_ids:
    raise RuntimeError('No frame records found in JSONL.')

all_objects = [obj for objs in frame_objects.values() for obj in objs]
unknown_dist = sum(1 for obj in all_objects if obj.distance_m is None)

print(f'Frames in JSONL: {len(frame_ids)}')
print(f'Frame range: {frame_ids[0]} -> {frame_ids[-1]}')
print(f'Total object entries: {len(all_objects)}')
print(f'Unknown distance objects: {unknown_dist} ({(100.0 * unknown_dist / max(len(all_objects), 1)):.2f}%)')

preview_n = min(2, len(frame_ids))
for fid in frame_ids[:preview_n]:
    preview = [(o.object_id, o.obj_class, o.distance_m, o.velocity_ms, o.position) for o in frame_objects[fid][:4]]
    print(f'Frame {fid}: {preview}')

In [ ]:
if not RGB_FRAMES_DIR.exists():
    raise FileNotFoundError(
        f'RGB frames directory not found: {RGB_FRAMES_DIR}\n'
        'Set SESSION_ID to a SANPO session that exists in data/sanpo/raw.'
    )

first_frame_path = None
for fid in frame_ids:
    first_frame_path = resolve_frame_path(fid)
    if first_frame_path is not None:
        break

if first_frame_path is None:
    raise RuntimeError('Could not find any frame image for JSONL frame_ids in this session.')

sample = cv2.imread(str(first_frame_path))
if sample is None:
    raise RuntimeError(f'Failed to read sample frame: {first_frame_path}')

h0, w0 = sample.shape[:2]
out_w, out_h = int(w0 * SCALE), int(h0 * SCALE)

session_fps = infer_session_fps()
stride = infer_stride(frame_ids)
render_fps = float(OUTPUT_FPS) if OUTPUT_FPS is not None else (session_fps / stride if session_fps else 10.0)
render_fps = max(1.0, float(render_fps))

OUTPUT_MP4.parent.mkdir(parents=True, exist_ok=True)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(str(OUTPUT_MP4), fourcc, render_fps, (out_w, out_h))

def score_color(value: float, max_value: float) -> tuple[int, int, int]:
    if max_value <= 0:
        return (180, 180, 180)
    r = max(0.0, min(1.0, value / max_value))
    # BGR: low risk ~ green, high risk ~ red
    b = int(40 * (1 - r))
    g = int(220 * (1 - r))
    rr = int(60 + 195 * r)
    return (b, g, rr)

rendered = 0
missing_frames = 0

for i, fid in enumerate(frame_ids):
    frame_path = resolve_frame_path(fid)
    if frame_path is None:
        missing_frames += 1
        continue

    img = cv2.imread(str(frame_path))
    if img is None:
        missing_frames += 1
        continue

    if SCALE != 1.0:
        img = cv2.resize(img, (out_w, out_h), interpolation=cv2.INTER_AREA)

    objects = frame_objects[fid]
    scored = [(obj, kinetic_score_custom(obj)) for obj in objects]
    scored.sort(key=lambda x: x[1], reverse=True)
    max_k = max((k for _, k in scored), default=1.0)

    overlay = img.copy()
    panel_h = min(out_h - 20, 95 + DRAW_TOP_N * 28)
    cv2.rectangle(overlay, (10, 10), (min(out_w - 10, 1150), panel_h), (15, 15, 15), -1)
    cv2.addWeighted(overlay, 0.58, img, 0.42, 0, img)

    header = f'Session: {SESSION_ID[:12]}...  Frame: {fid}  Objects: {len(scored)}  formula: severity*dir*|v|^{VELOCITY_EXP}/d^{DISTANCE_EXP}'
    cv2.putText(img, header, (22, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (230, 230, 230), 1, cv2.LINE_AA)

    y = 64
    for rank, (obj, k) in enumerate(scored[:DRAW_TOP_N], start=1):
        dist_txt = f'{obj.distance_m:.2f}m' if obj.distance_m is not None else 'unknown'
        line = f'{rank:02d}. {obj.object_id} | {obj.obj_class:<18} d={dist_txt:<8} v={obj.velocity_ms:>5.2f}  pos={obj.position:<9}  K*={k:>8.4f}'

        c = score_color(k, max_k)
        if rank == 1:
            cv2.rectangle(img, (18, y - 16), (min(out_w - 14, 1040), y + 8), (35, 35, 95), 1)

        cv2.putText(img, line, (24, y), cv2.FONT_HERSHEY_SIMPLEX, 0.52, c, 1, cv2.LINE_AA)

        bar_x = min(out_w - 360, 760)
        bar_w = min(280, out_w - bar_x - 20)
        if bar_w > 20:
            cv2.rectangle(img, (bar_x, y - 11), (bar_x + bar_w, y + 5), (60, 60, 60), -1)
            fill = int(bar_w * (k / max_k)) if max_k > 0 else 0
            if fill > 0:
                cv2.rectangle(img, (bar_x, y - 11), (bar_x + fill, y + 5), c, -1)

        y += 28

    writer.write(img)
    rendered += 1

    if i % 30 == 0:
        print(f'Rendered {i + 1}/{len(frame_ids)} JSONL frames...')

writer.release()

print('--- Render Complete ---')
print(f'Rendered frames: {rendered}')
print(f'Missing frames:  {missing_frames}')
print(f'Output video:    {OUTPUT_MP4}')
print(f'Render FPS:      {render_fps:.2f}')

In [ ]:
if OUTPUT_MP4.exists():
    display(Video(str(OUTPUT_MP4), embed=True, html_attributes='controls loop'))
else:
    print('Output video not found. Re-run the render cell above.')